## LIME(Local Model Agnostic Method)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.ensemble import RandomForestRegressor
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
# Load the dataset
data=pd.read_csv("data/abalone.data",
                 names=['sex', 'length', 'diameter', 'height',
                        'whole weight', 'shucked weight',
                        'viscera weight', 'shell weight', 'rings'])

In [ ]:
y=data['rings']
x=data[['sex', 'length', 'height', 'shucked weight',
        'viscera weight', 'shell weight']]

# Create dummy variables
x['sex.M']=[1 if s=='M' else 0 for s in x['sex']]
x['sex.F']=[1 if s=='F' else 0 for s in x['sex']]
x['sex.I']=[1 if s=='I' else 0 for s in x['sex']]

x=x.drop('sex', axis=1)

x.head(10)

In [ ]:
# Train the model
model=RandomForestRegressor()
model.fit(x.to_numpy(), y)

In [ ]:
# Get predictions
y_pred=model.predict(x)

fig, ax=plt.subplots(nrows=1, ncols=1, figsize=(8, 8))

plt.scatter(y, y_pred)
plt.plot([0, 30], [0, 30], color='r', linestyle='-', linewidth=2)

plt.ylabel('Predicted values', size=12)
plt.xlabel('Actual values', size=12)

### Local explanations

In [ ]:
# Create the explainer
explainer=LimeTabularExplainer(
    training_data=x.values,
    feature_names=x.columns,
    class_names=['rings'],
    mode='regression',
    random_state=101
)

In [ ]:
# Get the explanation of first row
exp=explainer.explain_instance(
    data_row=x.iloc[0],
    predict_fn=model.predict,
    labels=x.columns
)

# Display the explanation
from IPython.display import HTML, display

html=exp.as_html(show_table=True)
display(HTML(html))

In [ ]:
# Check calculations
exp_weight=[x[1] for x in exp.as_map()[1]]

print(np.mean(y_pred)+sum(exp_weight))
print(y_pred[0])

- In `LIME`, the sum of the weights and mean prediction will not be equal to the prediction for the given instance.

### Aggregating LIME weights

In [ ]:
# Combining LIME weights of many predictions using different charts
def return_weights(exp):
    exp_list=exp.as_map()[1]
    exp_list=sorted(exp_list, key=lambda x:x[0])
    exp_weights=[x[1] for x in exp_list]

    return exp_weights

In [ ]:
weights=[]

for row in x.values[0: 100]:
    exp=explainer.explain_instance(
        data_row=row,
        predict_fn=model.predict,
        labels=x.columns
    )

    exp_weight=return_weights(exp)
    weights.append(exp_weight)

# Create a Dataframe
lime_weights=pd.DataFrame(data=weights, columns=x.columns)

In [ ]:
print(np.shape(lime_weights))
lime_weights.head(10)

### Absolute Mean

In [ ]:
abs_mean=lime_weights.abs().mean(axis=0)
abs_mean=pd.DataFrame(data={'feature': abs_mean.index, 'abs_mean': abs_mean})
abs_mean=abs_mean.sort_values('abs_mean')

fig, ax=plt.subplots(nrows=1, ncols=1, figsize=(8, 4))

y_ticks=range(len(abs_mean))
y_labels=abs_mean.feature

plt.barh(y=y_ticks, width=abs_mean.abs_mean)
plt.yticks(ticks=y_ticks, labels=y_labels, size=12)
plt.title('')
plt.ylabel('')
plt.xlabel('Mean [Weight]', size=15)

- Features with large positive or negative lime weights have a large impact on a prediction, so features with `large absolute mean weight` will contribute more to the predictions. From the above abs mean bar graph, `shell weight` and `shucked weight` are most important when it comes to predicting `number of rings`.

### Feature Trend

In [ ]:
fig, ax=plt.subplots(nrows=1, ncols=1, figsize=(8, 3))

feature_weight=lime_weights['shell weight']
feature_value=x['shell weight'][0: 100]

plt.scatter(x=feature_value, y=feature_weight)

plt.ylabel('LIME weight', size=12)
plt.xlabel('Shell weight', size=12)

- We can see that as the shell weigth increases, the lime weight increases.

- The higher lime weight indicates that for a specific prediction, the feature value has increased the predicted number of rings.

### Beeswarm Plot

In [ ]:
fig, ax=plt.subplots(nrows=1, ncols=1, figsize=(8, 4))

y_ticks=range(len(abs_mean))
y_labels=abs_mean.feature

for i, feature in enumerate(y_labels):
    feature_weight=lime_weights[feature]
    feature_value=x[feature][0: 100]

    plt.scatter(
        x=feature_weight,
        y=[i]*len(feature_weight),
        c=feature_value,
        cmap='bwr',
        edgecolors='black',
        alpha=0.8
    )

plt.vlines(x=0, ymin=0, ymax=len(y_labels), colors='black', linestyles='--')
plt.colorbar(label='Feature Value', ticks=[])

plt.yticks(ticks=y_ticks, labels=y_labels, size=12)
plt.xlabel('LIME weight', size=15)